In [ ]:
# 准备训练、评测和播放录像的工具。
from pathlib import Path

import lightning as L
from IPython.display import Video, display

from training.evaluate import evaluate_airtime, load_checkpoint, record_video
from training.ppo import PPO, PPOData
from training.sim import SKILLS, DuckEnv, hardware

# 用随仓附带的出厂走路权重作为两组的共同起点。
checkpoint_path = Path("../../checkpoints/m05_factory_walk.pt")


In [ ]:
# 两组奖励：一组保留完整奖励，另一组只把滞空权重改为 0。
course_walk = SKILLS["course-walk"]
skills = {"full_reward": course_walk, "no_airtime": course_walk.with_reward_weights(air_time=0.0)}
# 两组都在同一硬件上训练，并用同一个任务录像。
walk = SKILLS["walk"]
hw = hardware.load("2026-11-26-hd1910")
# 从出厂走路权重接着训，两组各自重新载入。
def start_training(skill):
    # 固定随机种子，读回策略、价值网络和归一化统计量。
    L.seed_everything(1, verbose=False)
    net, checkpoint = load_checkpoint(checkpoint_path, "cuda")
    net.train()
    # 每组都是 2048 个并行环境，只切换奖励配置。
    env = DuckEnv(skill, hw, num_envs=2048, device="cuda", seed=1)
    # 课程表接着出厂权重训到的那一档。
    env.set_curriculum_step(checkpoint["common_step_counter"])
    # 打散回合进度，不让所有环境同时超时。
    env.reset()
    env.randomize_episode_starts()
    return env, net


In [ ]:
# 两组各训 300 轮，每轮用本组的最新策略重新采样。
runs = {}
for label, skill in skills.items():
    env, net = start_training(skill)
    try:
        # 各自构建 PPO，不在两组之间共用优化器状态。
        data = PPOData(env, net)
        model = PPO(net)
        trainer = L.Trainer(max_epochs=300, reload_dataloaders_every_n_epochs=1,
                            accelerator="gpu", devices=1, logger=False,
                            enable_checkpointing=False, enable_model_summary=False)
        trainer.fit(model, data)
        # 记下网络和课程进度，供录像与评测使用。
        runs[label] = net, env.unwrapped.common_step_counter
    finally:
        # 本组结束后释放仿真，再开始下一组。
        env.close()

In [ ]:
# 各录一段，同一机位、同一条前进指令。
for label, (net, clock) in runs.items():
    path = Path(f"m05_{label}.mp4")
    record_video(net, walk, hw, path, common_step_counter=clock)
    display(Video(str(path), embed=True))

In [ ]:
# 同一条前进指令下，比较左右脚腾空多久（毫秒）、抬得多高（毫米）。
{label: evaluate_airtime(net, hw, clock) for label, (net, clock) in runs.items()}